# Waaiya screen blur: fine-tune the person segmenter for silhouette covers

Person crops and instance masks from the **Open Images V7 train split** (Person / Man / Woman / Boy / Girl masks
and boxes, CC BY 4.0; images listed as CC BY 2.0), cut from images resized to the phone's 640-px screen frame, with
the runtime's detector box + 20% context. torchvision's LR-ASPP MobileNetV3-large (COCO with VOC labels, BSD-3) is
fine-tuned to one "this person" output at two input sizes. The MIAP extension is **not** used. Checkpoints are chosen
on a 1% holdout of train images; the validation and test splits stay untouched for the end-to-end scorer.

Outputs in `/kaggle/working/out`: `seg_<tag>.pt` (state_dict + size + mean/std + context), `seg_<tag>_eval.json`, logs.

In [ ]:
PREP_MINUTES = 45
TRAIN_MINUTES = 110   # per model when two GPUs train side by side; split in two on one GPU
IMAGES = 70000
SIZES = [('s192', 192, 128), ('s144', 144, 96)]
SEG_URL, BOX_URL, ZIPS_URL, WEIGHTS = 'https://storage.googleapis.com/openimages/v5/train-annotations-object-segmentation.csv', 'https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv', 'https://storage.googleapis.com/openimages/v5/train-masks/train-masks-{c}.zip', 'https://download.pytorch.org/models/lraspp_mobilenet_v3_large-d234d4ea.pth'
SEG_ROWS = 634241
PL = '$3=="/m/01g317" || $3=="/m/04yx4" || $3=="/m/03bt1vf" || $3=="/m/01bl7v" || $3=="/m/05r655"'

import os, subprocess, time, glob
CODE, TMP, OUT = "/kaggle/working/code", "/kaggle/tmp", "/kaggle/working/out"
for d in (CODE + "/models/vision", TMP, OUT):
    os.makedirs(d, exist_ok=True)

def run(cmd, cwd=CODE, env=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(["bash", "-o", "pipefail", "-c", cmd], cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
%%writefile /kaggle/working/code/models/vision/prep_seg_crops.py
"""Person crops + masks from the Open Images V7 *train* split, for the silhouette-cover segmenter (train_seg.py).

Open Images V7 instance masks (CC BY 4.0) of the person-like classes (Person, Man, Woman, Boy, Girl); the MIAP
extension is never used. Crops are cut the way the phone sees people: each image is resized to long side 640 (the
blur's screen frame), then each masked person box is cut with MARGIN extra on each side (train_seg.py jitters the
box and applies the runtime context inside that margin). Each crop has a label PNG of the same size:
    255  the person's own mask
    128  ignore: inside another person-like box (masked or not) and outside this person's mask
      0  background
so other people in the crop are neither taught as person nor as background.

    python models/vision/prep_seg_crops.py --seg-csv train-seg-person.csv --box-csv train-person-boxes.csv \
        --zips-url https://storage.googleapis.com/openimages/v5/train-masks/train-masks-{c}.zip --split train \
        --out /kaggle/tmp/segcrops [--images 60000] [--max-minutes 40]

--seg-csv: the object-segmentation CSV (or only its person-like rows, with the header).
--box-csv: the bbox CSV (or only its person-like rows, with the header): people without a mask become "ignore".
--zips-url: the mask archives, {c} = 0..9a..f; a local path works too (then the archives are kept).
Images are split by md5(image id): 1 in 100 goes to the *holdout* (checkpoints, never trained on).
Writes <out>/index.csv: image, label, holdout, bx0, by0, bx1, by1 (box inside the crop, 0..1), h_frame, crop, mask
"""
import argparse
import csv
import hashlib
import io
import os
import random
import shutil
import sys
import time
import urllib.request
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image

PERSONLIKE = {"/m/01g317": "person", "/m/04yx4": "man", "/m/03bt1vf": "woman", "/m/01bl7v": "boy", "/m/05r655": "girl"}
URL = "https://open-images-dataset.s3.amazonaws.com/{split}/{id}.jpg"
FRAME_LONG = 640
MARGIN = 0.35
CROP_LONG = 256
MIN_H = 12  # skip people shorter than this in the 640 frame (pixels)


def is_holdout(image_id):
    return int(hashlib.md5(image_id.encode()).hexdigest(), 16) % 100 == 0


def fetch(url, tries=3):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read()
        except Exception as e:  # noqa: BLE001 - network: retry, then give up
            if i == tries - 1:
                print(f"skip {url}: {e}", file=sys.stderr)
                return None
            time.sleep(1 + 2 * i)


def fetch_zip(src, dst):
    if os.path.exists(src):
        return src
    for i in range(3):
        try:
            with urllib.request.urlopen(src, timeout=120) as r, open(dst + ".part", "wb") as f:
                shutil.copyfileobj(r, f, 1 << 22)
            os.replace(dst + ".part", dst)
            return dst
        except Exception as e:  # noqa: BLE001
            print(f"zip retry {src}: {e}", file=sys.stderr)
            time.sleep(5)
    return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seg-csv", required=True)
    ap.add_argument("--box-csv", required=True)
    ap.add_argument("--zips-url", required=True)
    ap.add_argument("--split", default="train")
    ap.add_argument("--out", required=True)
    ap.add_argument("--images", type=int, default=60000, help="training images wanted (holdout images come on top)")
    ap.add_argument("--max-minutes", type=float, default=40, help="stop fetching images after this")
    ap.add_argument("--threads", type=int, default=16)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    t0 = time.time()

    masks = defaultdict(list)  # image -> [(mask path, label, x0, y0, x1, y1)]
    with open(a.seg_csv, newline="") as f:
        for r in csv.DictReader(f):
            if r["LabelName"] in PERSONLIKE:
                masks[r["ImageID"]].append((r["MaskPath"], PERSONLIKE[r["LabelName"]], float(r["BoxXMin"]),
                                            float(r["BoxYMin"]), float(r["BoxXMax"]), float(r["BoxYMax"])))
    ids = sorted(masks)
    hold = [i for i in ids if is_holdout(i)]
    rest = [i for i in ids if not is_holdout(i)]
    random.Random(a.seed).shuffle(rest)
    hold = hold[: max(200, a.images // 100)]
    chosen = hold + rest[: a.images]
    cset = set(chosen)
    print(f"{len(ids)} images with person masks; using {len(hold)} holdout + {len(chosen) - len(hold)} train",
          flush=True)

    boxes = defaultdict(list)
    with open(a.box_csv, newline="") as f:
        for r in csv.DictReader(f):
            if r["ImageID"] in cset and r["LabelName"] in PERSONLIKE:
                boxes[r["ImageID"]].append((float(r["XMin"]), float(r["YMin"]), float(r["XMax"]), float(r["YMax"])))

    # masks of the chosen images, archive by archive
    mdir = os.path.join(a.out, "rawmasks")
    os.makedirs(mdir, exist_ok=True)
    want = defaultdict(set)
    for i in chosen:
        for m in masks[i]:
            if not os.path.exists(os.path.join(mdir, m[0])):
                want[m[0][0]].add(m[0])
    for c, names in sorted(want.items()):
        z = fetch_zip(a.zips_url.format(c=c), os.path.join(a.out, f"masks-{c}.zip"))
        if not z:
            continue
        with zipfile.ZipFile(z) as zf:
            have = set(zf.namelist())
            for n in names & have:
                with zf.open(n) as s, open(os.path.join(mdir, n), "wb") as d:
                    shutil.copyfileobj(s, d)
        if z.startswith(a.out):
            os.remove(z)
        print(f"masks {c}: {len(names)} wanted, {time.time() - t0:.0f} s", flush=True)

    cdir = os.path.join(a.out, "crops")
    os.makedirs(cdir, exist_ok=True)
    deadline = t0 + a.max_minutes * 60

    def work(image_id):
        if time.time() > deadline:
            return []
        data = fetch(URL.format(split=a.split, id=image_id))
        if data is None:
            return []
        img = Image.open(io.BytesIO(data)).convert("RGB")
        s = FRAME_LONG / max(img.size)
        W, H = max(1, round(img.size[0] * s)), max(1, round(img.size[1] * s))
        img = img.resize((W, H), Image.BILINEAR)
        mine = masks[image_id]
        others = [(x0, y0, x1, y1) for (_, _, x0, y0, x1, y1) in mine] + boxes.get(image_id, [])
        lines = []
        for n, (mp, lab, x0, y0, x1, y1) in enumerate(mine):
            if (y1 - y0) * H < MIN_H or (x1 - x0) * W < 4:
                continue
            p = os.path.join(mdir, mp)
            if not os.path.exists(p):
                continue
            m = np.asarray(Image.open(p).convert("L").resize((W, H), Image.BILINEAR)) >= 128
            if not m.any():
                continue
            label = np.zeros((H, W), np.uint8)
            for (ox0, oy0, ox1, oy1) in others:
                if abs(ox0 - x0) < 1e-3 and abs(oy0 - y0) < 1e-3 and abs(ox1 - x1) < 1e-3 and abs(oy1 - y1) < 1e-3:
                    continue  # this person's own box
                label[int(oy0 * H):int(np.ceil(oy1 * H)), int(ox0 * W):int(np.ceil(ox1 * W))] = 128
            label[m] = 255
            bw, bh = (x1 - x0) * W, (y1 - y0) * H
            cx0, cy0 = max(0, int(x0 * W - MARGIN * bw)), max(0, int(y0 * H - MARGIN * bh))
            cx1, cy1 = min(W, int(np.ceil(x1 * W + MARGIN * bw))), min(H, int(np.ceil(y1 * H + MARGIN * bh)))
            crop, lc = img.crop((cx0, cy0, cx1, cy1)), Image.fromarray(label[cy0:cy1, cx0:cx1])
            k = CROP_LONG / max(crop.size)
            if k < 1:
                sz = (max(1, round(crop.size[0] * k)), max(1, round(crop.size[1] * k)))
                crop, lc = crop.resize(sz, Image.BILINEAR), lc.resize(sz, Image.NEAREST)
            name = f"{image_id}_{n}"
            crop.save(os.path.join(cdir, name + ".jpg"), quality=92)
            lc.save(os.path.join(cdir, name + ".png"))
            cw, ch = cx1 - cx0, cy1 - cy0
            lines.append([image_id, lab, int(is_holdout(image_id)), (x0 * W - cx0) / cw, (y0 * H - cy0) / ch,
                          (x1 * W - cx0) / cw, (y1 * H - cy0) / ch, y1 - y0, name + ".jpg", name + ".png"])
        return lines

    n_img = n_crop = 0
    with open(os.path.join(a.out, "index.csv"), "w", newline="") as f, ThreadPoolExecutor(a.threads) as ex:
        w = csv.writer(f)
        w.writerow(["image", "label", "holdout", "bx0", "by0", "bx1", "by1", "h_frame", "crop", "mask"])
        for lines in ex.map(work, chosen):
            w.writerows(lines)
            n_img += 1
            n_crop += len(lines)
            if n_img % 2000 == 0:
                f.flush()
                print(f"{n_img}/{len(chosen)} images, {n_crop} crops, {time.time() - t0:.0f} s", flush=True)
    shutil.rmtree(mdir, ignore_errors=True)
    print(f"done: {n_crop} crops from {n_img} images in {time.time() - t0:.0f} s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/models/vision/train_seg.py
"""Fine-tunes the silhouette-cover segmenter: torchvision LR-ASPP MobileNetV3-large (COCO with VOC labels, BSD-3)
on Open Images V7 person masks (prep_seg_crops.py), as one "this person" output.

The input is what the runtime gives it (engine/src/im_vision.c): the detector's box plus CONTEXT on each side,
stretched to H x W. Training jitters the annotated box (detector boxes are not exact) and samples the context inside
the crop's saved margin. Target: the person's own mask; pixels inside other people's boxes are ignored.
The new 1-channel head starts from the COCO model's person-minus-background logit.

    python models/vision/train_seg.py --data /kaggle/tmp/segcrops --weights lraspp_mobilenet_v3_large-d234d4ea.pth \
        --out out [--size 192 128] [--minutes 90] [--tag s192]

Writes <out>/seg_<tag>.pt (state_dict of the 1-class LRASPP, size, mean/std, context) and seg_<tag>_eval.json.
Checkpoints are chosen on the holdout images (1 in 100, never trained on).
"""
import argparse
import csv
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Dataset

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CONTEXT = 0.20  # runtime context around the detector box, each side, fraction of the box size
PERSON, BG = 15, 0


def build(weights=None):
    from torchvision.models.segmentation import lraspp_mobilenet_v3_large

    net = lraspp_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=21)
    if weights:
        net.load_state_dict(torch.load(weights, map_location="cpu", weights_only=True))
    head = net.classifier
    for name in ("low_classifier", "high_classifier"):
        old = getattr(head, name)
        new = torch.nn.Conv2d(old.in_channels, 1, 1)
        with torch.no_grad():
            new.weight.copy_(old.weight[PERSON:PERSON + 1] - old.weight[BG:BG + 1])
            new.bias.copy_(old.bias[PERSON:PERSON + 1] - old.bias[BG:BG + 1])
        setattr(head, name, new)
    return net


def low_res(net, x):
    """Logits at 1/8 of the input (the head's own resolution, before LR-ASPP's final upsampling)."""
    return net.classifier(net.backbone(x))


class Crops(Dataset):
    def __init__(self, root, rows, size, train):
        self.root, self.rows, self.size, self.train = root, rows, size, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(os.path.join(self.root, "crops", r["crop"])).convert("RGB")
        lab = Image.open(os.path.join(self.root, "crops", r["mask"]))
        W, H = img.size
        bx0, by0, bx1, by1 = (float(r[k]) for k in ("bx0", "by0", "bx1", "by1"))
        bw, bh = bx1 - bx0, by1 - by0
        rnd = random.Random(None if self.train else i)
        if self.train:  # a detector box: each edge moved by up to 8% of the size
            j = lambda: rnd.uniform(-0.08, 0.08)  # noqa: E731
            bx0, bx1, by0, by1 = bx0 + j() * bw, bx1 + j() * bw, by0 + j() * bh, by1 + j() * bh
            bw, bh = bx1 - bx0, by1 - by0
        ctx = CONTEXT
        x0, x1 = max(0.0, bx0 - ctx * bw) * W, min(1.0, bx1 + ctx * bw) * W
        y0, y1 = max(0.0, by0 - ctx * bh) * H, min(1.0, by1 + ctx * bh) * H
        if x1 - x0 < 2 or y1 - y0 < 2:
            x0, y0, x1, y1 = 0, 0, W, H
        h, w = self.size
        box = (int(x0), int(y0), max(int(x0) + 2, int(round(x1))), max(int(y0) + 2, int(round(y1))))
        img = img.resize((w, h), Image.BILINEAR, box=box)
        lab = lab.resize((w, h), Image.NEAREST, box=box)
        if self.train:
            if rnd.random() < 0.5:
                img, lab = img.transpose(Image.FLIP_LEFT_RIGHT), lab.transpose(Image.FLIP_LEFT_RIGHT)
            if rnd.random() < 0.2:
                img = img.filter(ImageFilter.GaussianBlur(rnd.uniform(0.3, 1.2)))
        x = torch.from_numpy(np.asarray(img, np.float32) / 255).permute(2, 0, 1)
        if self.train:  # brightness / contrast / saturation
            x = x * rnd.uniform(0.75, 1.25) + rnd.uniform(-0.08, 0.08)
            g = x.mean(0, keepdim=True)
            x = (g + (x - g) * rnd.uniform(0.7, 1.3)).clamp(0, 1)
        y = torch.from_numpy(np.asarray(lab, np.uint8).copy())
        return x, y


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--weights", required=True, help="torchvision lraspp_mobilenet_v3_large COCO checkpoint")
    ap.add_argument("--out", required=True)
    ap.add_argument("--size", type=int, nargs=2, default=[192, 128], metavar=("H", "W"))
    ap.add_argument("--tag", default="s192")
    ap.add_argument("--minutes", type=float, default=90)
    ap.add_argument("--batch", type=int, default=96)
    ap.add_argument("--lr", type=float, default=6e-4)
    ap.add_argument("--pos-weight", type=float, default=1.5, help="> 1 favours covering")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rows = list(csv.DictReader(open(os.path.join(a.data, "index.csv"), newline="")))
    if a.limit:
        rows = rows[: a.limit]
    tr = [r for r in rows if r["holdout"] == "0"]
    ho = [r for r in rows if r["holdout"] == "1"]
    print(f"train {len(tr)} crops, holdout {len(ho)}", flush=True)
    size = tuple(a.size)
    dl = DataLoader(Crops(a.data, tr, size, True), a.batch, shuffle=True, num_workers=a.workers, drop_last=True,
                    persistent_workers=a.workers > 0, pin_memory=dev == "cuda")
    dh = DataLoader(Crops(a.data, ho, size, False), 128, num_workers=a.workers)
    net = build(a.weights).to(dev)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    opt = torch.optim.AdamW(net.parameters(), lr=a.lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
    budget, warm = a.minutes * 60, 300
    t0, step, best, ep = time.time(), 0, -1.0, 0
    log = []

    def evaluate():
        net.eval()
        tp = fp = fn = 0.0
        rec_lo = []
        with torch.no_grad():
            for x, y in dh:
                x, y = x.to(dev), y.to(dev)
                p = torch.sigmoid(F.interpolate(low_res(net, (x - mean) / std), size=size, mode="bilinear",
                                                align_corners=False))[:, 0]
                valid, pos = y != 128, y == 255
                pr = p >= 0.5
                tp += (pr & pos).sum().item()
                fp += (pr & ~pos & valid).sum().item()
                fn += (~pr & pos).sum().item()
                for k in range(len(y)):  # per person: share of the mask above a low (covering) threshold
                    n = pos[k].sum().item()
                    if n:
                        rec_lo.append(((p[k] >= 0.2) & pos[k]).sum().item() / n)
        net.train()
        return {"iou": tp / max(1, tp + fp + fn), "recall@0.2": float(np.mean(rec_lo)),
                "p5_recall@0.2": float(np.percentile(rec_lo, 5))}

    print("start", evaluate(), flush=True)
    net.train()
    done = False
    while not done:
        ep += 1
        for x, y in dl:
            el = time.time() - t0
            if el > budget:
                done = True
                break
            lr = a.lr * min(1, (step + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, el / budget)))
            for g in opt.param_groups:
                g["lr"] = lr
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(dev, enabled=dev == "cuda"):
                lo = F.interpolate(low_res(net, (x - mean) / std), size=size, mode="bilinear", align_corners=False)
            lo = lo[:, 0].float()
            valid = (y != 128).float()
            tgt = (y == 255).float()
            w = valid * (1 + (a.pos_weight - 1) * tgt)
            loss = (F.binary_cross_entropy_with_logits(lo, tgt, reduction="none") * w).sum() / valid.sum().clamp(min=1)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            step += 1
            if step % 200 == 0:
                print(f"ep {ep} step {step} loss {loss.item():.4f} lr {lr:.2e} {el / 60:.1f} min", flush=True)
        ev = evaluate()
        ev.update(epoch=ep, step=step, minutes=(time.time() - t0) / 60)
        log.append(ev)
        print("eval", ev, flush=True)
        if ev["iou"] > best:
            best = ev["iou"]
            torch.save({"state_dict": net.state_dict(), "size": list(size), "mean": MEAN, "std": STD,
                        "context": CONTEXT, "eval": ev}, os.path.join(a.out, f"seg_{a.tag}.pt"))
    json.dump(log, open(os.path.join(a.out, f"seg_{a.tag}_eval.json"), "w"), indent=1)
    print("best", best, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
PY = f"{TMP}/venv/bin/python"
if not os.path.exists(PY):
    run("pip install -q uv")
    run(f"uv venv -q -p 3.10 {TMP}/venv")
    run(f"uv pip install -q -p {PY} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121")
    run(f"uv pip install -q -p {PY} numpy==1.26.4 pillow==11.0.0 setuptools==69.5.1")  # torch 2.1 imports pkg_resources
run(f"{PY} -c 'import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.device_count(), \"GPUs\")'")
run("nproc; free -g; df -h /kaggle/tmp /kaggle/working")
W = f"{TMP}/lraspp_coco.pth"
if not os.path.exists(W):
    run(f"curl -sfL {WEIGHTS} -o {W}")

In [ ]:
# mask index (~800 MB) and box CSV (~2.3 GB): stream them, keep only the person-like rows
SEG = f"{TMP}/train-seg-person.csv"
BOX = f"{TMP}/train-person-boxes.csv"
if not os.path.exists(SEG):
    run(f"curl -sfL {SEG_URL} | awk -F, 'NR==1 || {PL}' > {SEG}.part && mv {SEG}.part {SEG}")
if not os.path.exists(BOX):
    run(f"curl -sfL {BOX_URL} | awk -F, 'NR==1 || {PL}' > {BOX}.part && mv {BOX}.part {BOX}")
n = sum(1 for _ in open(SEG)) - 1
print(n, "mask rows;", sum(1 for _ in open(BOX)) - 1, "box rows")
assert n == SEG_ROWS, f"expected {SEG_ROWS} rows, got {n}"

In [ ]:
DATA = f"{TMP}/segcrops"
if not os.path.exists(f"{DATA}/index.csv") or os.path.exists(f"{DATA}/rawmasks"):
    run(f"{PY} models/vision/prep_seg_crops.py --seg-csv {SEG} --box-csv {BOX} --zips-url {ZIPS_URL} --split train "
        f"--out {DATA} --images {IMAGES} --max-minutes {PREP_MINUTES} --threads 24 2>{TMP}/prep_err.log")
run(f"tail -n 3 {TMP}/prep_err.log; grep -c skip {TMP}/prep_err.log || true; du -sh {DATA}")
run(f"cut -d, -f2,3 {DATA}/index.csv | sort | uniq -c")

In [ ]:
ngpu = int(subprocess.check_output([PY, "-c", "import torch; print(torch.cuda.device_count())"]).decode())
ncpu = os.cpu_count()

def cmd(tag, h, w, minutes, workers):
    return [PY, "models/vision/train_seg.py", "--data", DATA, "--weights", W, "--out", OUT, "--size", str(h), str(w),
            "--tag", tag, "--minutes", str(minutes), "--workers", str(workers)]

if ngpu >= 2:   # one model per GPU, side by side
    procs = []
    for k, (tag, h, w) in enumerate(SIZES):
        log = open(f"{OUT}/seg_{tag}.log", "w")
        procs.append((tag, subprocess.Popen(cmd(tag, h, w, TRAIN_MINUTES, max(2, ncpu // 2)), cwd=CODE, stdout=log,
                                            stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(k)})))
    while any(p.poll() is None for _, p in procs):
        time.sleep(300)
        for tag, _ in procs:
            lines = open(f"{OUT}/seg_{tag}.log").read().strip().splitlines()
            print(tag, "|", lines[-1][:300] if lines else "(no output yet)", flush=True)
    for tag, p in procs:
        print(tag, "exit", p.returncode)
else:
    for tag, h, w in SIZES:
        run(" ".join(cmd(tag, h, w, TRAIN_MINUTES / 2, ncpu)) + f" 2>&1 | tee {OUT}/seg_{tag}.log | grep -v '^ep '")

In [ ]:
run(f"ls -la {OUT}; grep -h '^eval\\|^best' {OUT}/*.log || true")